In [ ]:
# ============================================================
# Experiment 2 – Statistical Analyses and Plots
# ============================================================

# Description:
# Statistical analyses and visualisations for
# Experiment 2 (Explicit Task).
#
# Analyses:
# 1. Descriptive statistics
# 2. Baseline within-participant model
# 3. Task-adjusted model
# 4. Time-on-task model
# 5. Between-participant model
#
# Figures:
# Figure 1  - RT distribution across tasks
# Figure 2  - TFF distribution across tasks
# Figure 4  - Within-participant TFF–RT relationship
# Figure 6  - Task effects on RT
# Figure 8  - Time-on-task moderation
# Figure 10 - Between-participant relationship
# ============================================================

# ============================================================
# Imports
# ============================================================

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf

# ============================================================
# Create Output Folders
# ============================================================

os.makedirs("Results", exist_ok=True)

os.makedirs("Figures", exist_ok=True)

# ============================================================
# Load Datasets
# ============================================================

df_exp1 = pd.read_csv(
    "Processed_Data/Exp1_ready_TFF.csv"
)

df_exp2 = pd.read_csv(
    "Processed_Data/Exp2_ready_TFF.csv"
)

# ============================================================
# Add Experiment Labels
# ============================================================

df_exp1["experiment"] = "Implicit"

df_exp2["experiment"] = "Explicit"

# ============================================================
# Combined Dataset for Descriptive Plots
# ============================================================

df_combined = pd.concat(
    [df_exp1, df_exp2],
    ignore_index=True
)

# ============================================================
# Use Experiment 2 for Inferential Analyses
# ============================================================

df = df_exp2.copy()

print("=" * 60)
print("Experiment 2 Dataset Loaded")
print("=" * 60)

print(f"Participants: {df['participant'].nunique()}")
print(f"Trials: {len(df)}")

# ============================================================
# Prepare Variables
# ============================================================

# ------------------------------------------------------------
# Center TFF within participant
# ------------------------------------------------------------

df["mean_TFF"] = (

    df.groupby("participant")["TFF_ms"]
      .transform("mean")
)

df["TFF_c"] = (
    df["TFF_ms"] - df["mean_TFF"]
)

# ------------------------------------------------------------
# Standardize trial order
# ------------------------------------------------------------

df["trial_z"] = (

    df.groupby("participant")["trial_index"]
      .transform(
          lambda x:
          (x - x.mean()) / x.std(ddof=0)
      )
)

# ============================================================
# Descriptive Statistics
# ============================================================

desc = pd.DataFrame({

    "Variable": ["RT_ms", "TFF_ms"],

    "Mean": [
        df["RT_ms"].mean(),
        df["TFF_ms"].mean()
    ],

    "SD": [
        df["RT_ms"].std(ddof=1),
        df["TFF_ms"].std(ddof=1)
    ],

    "Min": [
        df["RT_ms"].min(),
        df["TFF_ms"].min()
    ],

    "Max": [
        df["RT_ms"].max(),
        df["TFF_ms"].max()
    ]
})

desc = desc.round(3)

print("\nDescriptive Statistics")
print(desc)

desc.to_csv(
    "Results/Exp2_Descriptive_Statistics.csv",
    index=False
)

# ============================================================
# Plot Style
# ============================================================

sns.set_theme(style="whitegrid")

palette = {
    "Implicit": "skyblue",
    "Explicit": "#C8A2C8"
}

# ============================================================
# FIGURE 1
# RT Distribution Across Tasks
# ============================================================

plt.figure(figsize=(8,6))

sns.violinplot(
    data=df_combined,

    x="experiment",

    y="RT_ms",

    palette=palette,

    inner="box"
)

plt.title(
    "Reaction Time Across Tasks"
)

plt.ylabel("Reaction Time (ms)")

plt.xlabel("Task")

plt.tight_layout()

plt.savefig(
    "Figures/Figure1_RT_Distribution.png",
    dpi=300
)

plt.show()

# ============================================================
# FIGURE 2
# TFF Distribution Across Tasks
# ============================================================

plt.figure(figsize=(8,6))

sns.violinplot(
    data=df_combined,

    x="experiment",

    y="TFF_ms",

    palette=palette,

    inner="box"
)

plt.title(
    "Time to First Fixation Across Tasks"
)

plt.ylabel("TFF (ms)")

plt.xlabel("Task")

plt.tight_layout()

plt.savefig(
    "Figures/Figure2_TFF_Distribution.png",
    dpi=300
)

plt.show()

# ============================================================
# MODEL 1
# Baseline Within-Participant Model
# logRT ~ TFF_c + (1 | participant)
# ============================================================

print("\n" + "=" * 60)
print("MODEL 1: Within-Participant Model")
print("=" * 60)

model1 = smf.mixedlm(
    "logRT ~ TFF_c",
    data=df,
    groups=df["participant"]
)

result1 = model1.fit(reml=True)

print(result1.summary())

with open(
    "Results/Exp2_Model1.txt",
    "w"
) as f:

    f.write(result1.summary().as_text())

# ============================================================
# FIGURE 4
# Within-Participant TFF–RT Relationship
# ============================================================

plt.figure(figsize=(7,5))

sns.regplot(
    data=df,

    x="TFF_c",

    y="logRT",

    scatter_kws={
        "alpha": .08,
        "s": 12
    },

    line_kws={
        "linewidth": 2
    }
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Centered TFF within participant (ms)"
)

plt.ylabel("logRT")

plt.title(
    "Within-Participant TFF–RT Relationship: Explicit Task"
)

plt.tight_layout()

plt.savefig(
    "Figures/Figure4_WithinParticipant_TFF_RT.png",
    dpi=300
)

plt.show()

# ============================================================
# MODEL 2
# Task-Adjusted Model
# logRT ~ TFF_c + valence + side + congruence
# ============================================================

print("\n" + "=" * 60)
print("MODEL 2: Task-Adjusted Model")
print("=" * 60)

model2 = smf.mixedlm(
    "logRT ~ TFF_c + valence + side + congruence",
    data=df,
    groups=df["participant"]
)

result2 = model2.fit(reml=True)

print(result2.summary())

with open(
    "Results/Exp2_Model2_TaskAdjusted.txt",
    "w"
) as f:

    f.write(result2.summary().as_text())

# ============================================================
# FIGURE 6
# Task Effects on RT
# ============================================================

plot_df = (
    df.groupby(
        ["participant", "congruence", "side"],
        as_index=False
    )
    .agg(
        mean_logRT=("logRT", "mean")
    )
)

plot_df["congruence"] = plot_df["congruence"].map({
    -1: "Incongruent",
     1: "Congruent"
})

plot_df["side"] = plot_df["side"].map({
    -1: "Left",
     1: "Right"
})

plt.figure(figsize=(7,5))

sns.boxplot(
    data=plot_df,

    x="congruence",

    y="mean_logRT",

    hue="side"
)

sns.stripplot(
    data=plot_df,

    x="congruence",

    y="mean_logRT",

    hue="side",

    dodge=True,

    alpha=.25,

    size=3,

    legend=False
)

plt.xlabel("Congruence")

plt.ylabel("Mean logRT")

plt.title(
    "Task Effects on RT: Explicit Task"
)

plt.tight_layout()

plt.savefig(
    "Figures/Figure6_TaskEffects_RT.png",
    dpi=300
)

plt.show()

# ============================================================
# MODEL 3
# Time-on-Task Model
# logRT ~ TFF_c * trial_z
# ============================================================

print("\n" + "=" * 60)
print("MODEL 3: Time-on-Task Model")
print("=" * 60)

model3 = smf.mixedlm(
    "logRT ~ TFF_c * trial_z",
    data=df,
    groups=df["participant"]
)

result3 = model3.fit(reml=True)

print(result3.summary())

with open(
    "Results/Exp2_Model3_TimeOnTask.txt",
    "w"
) as f:

    f.write(result3.summary().as_text())

# ============================================================
# FIGURE 8
# Time-on-Task Effects
# ============================================================

x_vals = np.linspace(-40, 40, 100)

trial_levels = {

    "Early trials": -1,

    "Mid-task trials": 0,

    "Late trials": 1
}

plt.figure(figsize=(7,5))

for label, trial_level in trial_levels.items():

    pred_df = pd.DataFrame({

        "TFF_c": x_vals,

        "trial_z": trial_level
    })

    pred_df["pred"] = result3.predict(pred_df)

    plt.plot(
        pred_df["TFF_c"],
        pred_df["pred"],

        linewidth=2,

        label=label
    )

plt.axvline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(
    "Centered TFF within participant (ms)"
)

plt.ylabel("Predicted logRT")

plt.title(
    "Time-on-Task Effects on the TFF–RT Relationship"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "Figures/Figure8_TimeOnTask.png",
    dpi=300
)

plt.show()

# ============================================================
# MODEL 4
# Between-Participant Model
# mean logRT ~ mean TFF
# ============================================================

participant_df = (

    df.groupby("participant", as_index=False)
      .agg(

          mean_TFF=("TFF_ms", "mean"),

          mean_logRT=("logRT", "mean")
      )
)

print("\n" + "=" * 60)
print("MODEL 4: Between-Participant Model")
print("=" * 60)

model4 = smf.ols(
    "mean_logRT ~ mean_TFF",
    data=participant_df
).fit()

print(model4.summary())

with open(
    "Results/Exp2_Model4_BetweenParticipant.txt",
    "w"
) as f:

    f.write(model4.summary().as_text())

# ============================================================
# FIGURE 10
# Between-Participant TFF–RT Relationship
# ============================================================

plt.figure(figsize=(7,5))

sns.regplot(
    data=participant_df,

    x="mean_TFF",

    y="mean_logRT",

    scatter_kws={
        "alpha": .5,
        "s": 30
    },

    line_kws={
        "linewidth": 2
    }
)

plt.xlabel("Participant mean TFF (ms)")

plt.ylabel("Participant mean logRT")

plt.title(
    "Between-Participant TFF–RT Relationship: Explicit Task"
)

plt.tight_layout()

plt.savefig(
    "Figures/Figure10_BetweenParticipant.png",
    dpi=300
)

plt.show()
